<a href="https://colab.research.google.com/github/arturexxx/1erProyecto/blob/master/notebooks/Agente_RETCC_Alura_Desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================================================================
# PROYECTO: AGENTE DE IA PARA CONSULTAS DEL REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL (RETCC)
# ==========================================================================================================
# Descripción:
# Este proyecto implementa un sistema RAG (Retrieval-Augmented
# Generation) capaz de comprender el contenido de múltiples
# documentos PDF relacionados con el Registro Nacional de
# Trabajadores de Construcción Civil (RETCC) en el Estado Peruano.
#
# El agente podrá responder preguntas sobre:
# - Reglamento del RETCC.
# - Requisitos para la inscripción del Carné RETCC.
# - Requisitos para Renovacion de Carné RETCC.
#
# Documentos utilizados:
# 1. Reglamento_Inscripcion.pdf
# 2. Requisitos_Inscripcion_Carnet_RETCC.pdf
#
# Tecnologías:
# - Python
# - Google Colab
# - LangChain
# - Hugging Face (Embeddings)
# - FAISS (Base Vectorial)
# - Google Gemini (LLM)
#
# Objetivo:
# Construir un asistente inteligente que consulte la información
# de los documentos y responda preguntas utilizando RAG.
# ============================================================

print("🚀 Iniciando el proyecto: Agente IA RAG para documentos del RETCC")

🚀 Iniciando el proyecto: Agente IA RAG para documentos del RETCC


In [2]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-dotenv \
    gradio==5.49.1

In [3]:
from google.colab import files

archivos_subidos = files.upload()

Saving CanalesdeAtencionRETCC_costos.pdf to CanalesdeAtencionRETCC_costos.pdf
Saving Requisitos_Inscripcion_Carnet_RETCC.pdf to Requisitos_Inscripcion_Carnet_RETCC.pdf
Saving Reglamento_Incripcion.pdf to Reglamento_Incripcion.pdf


In [4]:
lista_pdfs = list(archivos_subidos.keys())

print(lista_pdfs)

['CanalesdeAtencionRETCC_costos.pdf', 'Requisitos_Inscripcion_Carnet_RETCC.pdf', 'Reglamento_Incripcion.pdf']


In [5]:
from langchain_community.document_loaders import PyPDFLoader

documentos = []

for pdf in lista_pdfs:
    print(f"Cargando: {pdf}")

    loader = PyPDFLoader(pdf)
    docs = loader.load()

    documentos.extend(docs)

print("\n-----------------------------")
print(f"Total de páginas cargadas: {len(documentos)}")

/tmp/ipykernel_51176/3316716.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Cargando: CanalesdeAtencionRETCC_costos.pdf
Cargando: Requisitos_Inscripcion_Carnet_RETCC.pdf
Cargando: Reglamento_Incripcion.pdf

-----------------------------
Total de páginas cargadas: 11


In [6]:
import re

def limpiar_texto(texto: str) -> str:
    """
    Limpia el texto extraído del PDF sin eliminar información importante.
    """

    # Reemplaza espacios especiales
    texto = texto.replace("\xa0", " ")

    # Une palabras separadas por guion al final de una línea
    # Ejemplo: "traba-\njador" -> "trabajador"
    texto = re.sub(r"(\w)-\n(\w)", r"\1\2", texto)

    # Reemplaza saltos simples de línea por espacios
    texto = re.sub(r"(?<!\n)\n(?!\n)", " ", texto)

    # Reduce múltiples saltos de línea
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    # Reduce múltiples espacios
    texto = re.sub(r"[ \t]+", " ", texto)

    # Elimina espacios al inicio y final
    return texto.strip()


print("Función de limpieza creada correctamente.")

Función de limpieza creada correctamente.


In [7]:
documentos_limpios = []

for documento in documentos:
    documento.page_content = limpiar_texto(documento.page_content)
    documentos_limpios.append(documento)

print(f"Páginas limpiadas: {len(documentos_limpios)}")

Páginas limpiadas: 11


In [8]:
print(documentos_limpios[0].page_content[:2000])

REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL: MODALIDADES: (D.S. 009-2016-TR y sus modificaciones): - PRESENCIAL (Ver las direcciones consignadas en el cuadro) - VIRTUAL: el link está activo las 24 horas ( https://portal.trabajo.gob.pe/retcc-virtual/ ) RECUERDA QUE EL RECOJO DEL CARNÉ RETCC, DE AMBAS MODALIDADES ES PRESENCIAL en las siguientes direcciones del presente cuadro RECUERDE: PARA EL DUPLICADO DEL CARNE RETCC, EN TODAS LAS DRTPE /GRTPE SE LE SOLICITA CONSTANCIA POLICIAL DE PERDIDA Y ROBO (DECRETO SUPREMO N° 0016-2019-TR) Y EL PAGO (siempre y cuando este establecido en el TUPA de la región) GERENCIA/DIRECCIÓ N REGIONALES DE TRABAJO Y PROMOCIÓN DEL EMPLEO/ ZONAL DIRECCIÓN CORREO Y/O TELÉFONO PARA CONSULTAS HORARIO DE ATENCIÓN EN OFICINA TASA DE PAGO SOLAMENTE PARA EL TRAMITE DEL DUPLICADO AMAZONAS DUPLICADO GRATUITO CHACHAPOYAS JR. SANTA ANA 1259 Cel: 921 170 277 lunes a viernes 8:00 am a 1:00 pm 2:30 pm a 5:30 pm BAGUA JR. JOSÉ GÁLVEZ 225 REFERENCIA DEL OVALO A 2 CUA

In [9]:
documentos_limpios = [
    documento
    for documento in documentos_limpios
    if documento.page_content.strip()
]

print(f"Páginas con contenido: {len(documentos_limpios)}")

Páginas con contenido: 11


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " ",
        ""
    ]
)

print("Divisor configurado correctamente.")

Divisor configurado correctamente.


In [11]:
fragmentos = text_splitter.split_documents(documentos_limpios)

print(f"Páginas procesadas: {len(documentos_limpios)}")
print(f"Fragmentos generados: {len(fragmentos)}")

Páginas procesadas: 11
Fragmentos generados: 52


In [12]:
for indice, fragmento in enumerate(fragmentos):
    pagina_langchain = fragmento.metadata.get("page", 0)

    fragmento.metadata["chunk_id"] = indice
    fragmento.metadata["pagina_real"] = pagina_langchain + 1

print("Metadatos de trazabilidad agregados.")

Metadatos de trazabilidad agregados.


In [13]:
cantidad_mostrar = min(5, len(fragmentos))

for indice in range(cantidad_mostrar):
    fragmento = fragmentos[indice]

    print("=" * 80)
    print(f"Fragmento: {fragmento.metadata['chunk_id']}")
    print(f"Página: {fragmento.metadata['pagina_real']}")
    print(f"Caracteres: {len(fragmento.page_content)}")
    print("-" * 80)
    print(fragmento.page_content[:700])
    print()

Fragmento: 0
Página: 1
Caracteres: 954
--------------------------------------------------------------------------------
REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL: MODALIDADES: (D.S. 009-2016-TR y sus modificaciones): - PRESENCIAL (Ver las direcciones consignadas en el cuadro) - VIRTUAL: el link está activo las 24 horas ( https://portal.trabajo.gob.pe/retcc-virtual/ ) RECUERDA QUE EL RECOJO DEL CARNÉ RETCC, DE AMBAS MODALIDADES ES PRESENCIAL en las siguientes direcciones del presente cuadro RECUERDE: PARA EL DUPLICADO DEL CARNE RETCC, EN TODAS LAS DRTPE /GRTPE SE LE SOLICITA CONSTANCIA POLICIAL DE PERDIDA Y ROBO (DECRETO SUPREMO N° 0016-2019-TR) Y EL PAGO (siempre y cuando este establecido en el TUPA de la región) GERENCIA/DIRECCIÓ N REGIONALES DE TRABAJO Y PROMOCIÓN DEL EMPLEO/ ZONAL DIRECCIÓN C

Fragmento: 1
Página: 1
Caracteres: 868
--------------------------------------------------------------------------------
. SANTA ANA 1259 Cel: 921 170 277 lunes a viernes 8:00 am 

In [14]:
fragmentos_vacios = [
    fragmento
    for fragmento in fragmentos
    if not fragmento.page_content.strip()
]

longitudes = [
    len(fragmento.page_content)
    for fragmento in fragmentos
]

print(f"Total de fragmentos: {len(fragmentos)}")
print(f"Fragmentos vacíos: {len(fragmentos_vacios)}")

if longitudes:
    print(f"Tamaño mínimo: {min(longitudes)} caracteres")
    print(f"Tamaño máximo: {max(longitudes)} caracteres")
    print(f"Tamaño promedio: {sum(longitudes) / len(longitudes):.2f} caracteres")

Total de fragmentos: 52
Fragmentos vacíos: 0
Tamaño mínimo: 13 caracteres
Tamaño máximo: 1000 caracteres
Tamaño promedio: 791.77 caracteres


In [15]:
!pip install -qU langchain-huggingface sentence-transformers

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

print("HuggingFaceEmbeddings importado correctamente.")

HuggingFaceEmbeddings importado correctamente.


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

In [18]:
MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print(f"Modelo seleccionado: {MODELO_EMBEDDINGS}")

Modelo seleccionado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [19]:
import torch

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo seleccionado: {dispositivo}")

if dispositivo == "cuda":
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
else:
    print("Se utilizará CPU.")

Dispositivo seleccionado: cpu
Se utilizará CPU.


In [20]:
modelo_embeddings = HuggingFaceEmbeddings(
    model_name=MODELO_EMBEDDINGS,
    model_kwargs={
        "device": dispositivo
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32
    }
)

print("Modelo de embeddings cargado correctamente.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embeddings cargado correctamente.


In [21]:
import numpy as np

def similitud_coseno(vector_a, vector_b):
    vector_a = np.array(vector_a)
    vector_b = np.array(vector_b)

    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) *
        np.linalg.norm(vector_b)
    )

In [22]:
from langchain_community.vectorstores import FAISS

print("FAISS importado correctamente.")

FAISS importado correctamente.


In [23]:
vectorstore = FAISS.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings
)

print("Base vectorial creada correctamente.")

Base vectorial creada correctamente.


In [24]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [25]:
!pip install -qU langchain-google-genai

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI

print("Integración de Gemini importada correctamente.")

Integración de Gemini importada correctamente.


In [27]:
from google.colab import userdata
import os

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "No se encontró GOOGLE_API_KEY en los secretos de Colab."
    )

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API Key de Gemini cargada correctamente.")

API Key de Gemini cargada correctamente.


In [28]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_retries=1
)

print("Modelo Gemini configurado correctamente.")

Modelo Gemini configurado correctamente.


In [29]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

print("Retriever configurado correctamente.")

Retriever configurado correctamente.


In [30]:
from langchain_core.prompts import ChatPromptTemplate

prompt_rag = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Eres un asistente especializado en responder preguntas sobre
los documentos del Registro Nacional de Trabajadores de
Construcción Civil, RETCC.

Debes responder utilizando exclusivamente la información
contenida en el contexto proporcionado.

Reglas obligatorias:

1. No inventes información.
2. No utilices conocimientos externos.
3. Si la respuesta no se encuentra en el contexto, responde:
   "No encontré esa información en los documentos proporcionados."
4. Responde en español.
5. Utiliza un lenguaje claro, ordenado y formal.
6. Cuando existan requisitos o pasos, preséntalos de manera enumerada.
7. No menciones información que no aparezca en el contexto.
8. No menciones archivos, páginas, documentos recuperados ni fuentes.
9. Responde únicamente la consulta del usuario.
            """
        ),
        (
            "human",
            """
Contexto recuperado de los documentos:

{contexto}

Pregunta del usuario:

{pregunta}
            """
        )
    ]
)

print("Prompt del RAG creado correctamente.")

Prompt del RAG creado correctamente.


In [31]:
def formatear_contexto(documentos):
    bloques = []

    for numero, documento in enumerate(documentos, start=1):
        archivo = documento.metadata.get(
            "source",
            "Documento desconocido"
        )

        pagina = documento.metadata.get(
            "pagina_real",
            documento.metadata.get("page", 0) + 1
        )

        bloque = f"""
FRAGMENTO {numero}
Archivo: {archivo}
Página: {pagina}

Contenido:
{documento.page_content}
"""

        bloques.append(bloque.strip())

    return "\n\n" + ("\n\n" + "-" * 80 + "\n\n").join(bloques)

In [32]:
def responder_pregunta(pregunta: str) -> dict:
    """
    Recupera información desde FAISS y genera una respuesta
    mediante Gemini utilizando exclusivamente los PDF.
    """
    if not pregunta or not pregunta.strip():
        return {
            "respuesta": "Debe ingresar una pregunta.",
            "fuentes": []
        }

    pregunta = pregunta.strip()

    # 1. Recuperar fragmentos desde FAISS
    documentos = retriever.invoke(pregunta)

    # 2. Preparar el contexto
    contexto = formatear_contexto(documentos)

    # 3. Construir el prompt
    mensajes = prompt_rag.invoke(
        {
            "contexto": contexto,
            "pregunta": pregunta
        }
    )

    # 4. Enviar el prompt a Gemini
    resultado = llm.invoke(mensajes)

    # 5. Preparar las fuentes sin duplicados
    fuentes = []

    for documento in documentos:
        fuente = {
            "archivo": documento.metadata.get(
                "source",
                "Documento desconocido"
            ),
            "pagina": documento.metadata.get(
                "pagina_real",
                documento.metadata.get("page", 0) + 1
            )
        }

        if fuente not in fuentes:
            fuentes.append(fuente)

    return {
        "respuesta": resultado.content,
        "fuentes": fuentes,
        "documentos": documentos
    }

In [33]:
def preguntar_al_agente(pregunta: str) -> None:
    resultado = responder_pregunta(pregunta)

    print("=" * 90)
    print("PREGUNTA")
    print("=" * 90)
    print(pregunta)

    print("\n" + "=" * 90)
    print("RESPUESTA")
    print("=" * 90)
    print(resultado["respuesta"])

    print("\n" + "=" * 90)
    print("FUENTES RECUPERADAS")
    print("=" * 90)

    if resultado["fuentes"]:
        for fuente in resultado["fuentes"]:
            print(
                f"- {fuente['archivo']}, "
                f"página {fuente['pagina']}"
            )
    else:
        print("No se recuperaron fuentes.")

In [34]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("Componentes de memoria importados correctamente.")


Componentes de memoria importados correctamente.


In [35]:
almacen_conversaciones = {}

def obtener_historial(session_id: str) -> InMemoryChatMessageHistory:
    """
    Recupera el historial de una sesión.
    Si no existe, crea uno nuevo.
    """

    if session_id not in almacen_conversaciones:
        almacen_conversaciones[session_id] = InMemoryChatMessageHistory()

    return almacen_conversaciones[session_id]


print("Almacén de conversaciones creado correctamente.")

Almacén de conversaciones creado correctamente.


In [36]:
prompt_reformular = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Tu tarea es reformular la última pregunta del usuario como una
            pregunta independiente y completa, utilizando el historial de
            conversación.

            Reglas:

            1. No respondas la pregunta.
            2. Devuelve únicamente la pregunta reformulada.
            3. Conserva el idioma español.
            4. No agregues información que no aparezca en la conversación.
            5. Si la pregunta ya es independiente, devuélvela sin modificarla.

            Ejemplo:

            Historial:
            Usuario: ¿Cuáles son los requisitos para inscribirse en el RETCC?

            Última pregunta:
            ¿Y cuánto demora?

            Pregunta reformulada:
            ¿Cuánto demora el trámite de inscripción en el RETCC?
            """
        ),
        MessagesPlaceholder(variable_name="historial"),
        (
            "human",
            "{pregunta}"
        )
    ]
)

print("Prompt de reformulación creado correctamente.")

Prompt de reformulación creado correctamente.


In [37]:
def reformular_pregunta(
    pregunta: str,
    historial: list
) -> str:
    """
    Convierte una pregunta continuada en una pregunta independiente.
    """

    # Si no existe historial, no es necesario reformular
    if not historial:
        return pregunta.strip()

    mensajes = prompt_reformular.invoke(
        {
            "historial": historial,
            "pregunta": pregunta.strip()
        }
    )

    resultado = llm.invoke(mensajes)

    pregunta_reformulada = resultado.content.strip()

    return pregunta_reformulada

In [38]:
prompt_rag_con_memoria = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Eres un asistente especializado en los documentos del Registro
            Nacional de Trabajadores de Construcción Civil, RETCC.

            Debes responder utilizando exclusivamente la información contenida
            en el contexto recuperado de los documentos.

            Reglas obligatorias:

            1. No inventes información.
            2. No uses conocimientos externos.
            3. Si la respuesta no aparece en el contexto, responde exactamente:
              "No encontré esa información en los documentos proporcionados."
            4. Responde siempre en español.
            5. Utiliza un lenguaje claro, formal y ordenado.
            6. Cuando existan requisitos o pasos, preséntalos numerados.
            7. Puedes utilizar el historial para comprender la conversación,
              pero los datos de la respuesta deben proceder del contexto.
            8. No afirmes que un requisito existe si no figura en los documentos.
            9. No menciones archivos, páginas, documentos recuperados ni fuentes.
            10. Responde únicamente la consulta del usuario.
            """
        ),
        MessagesPlaceholder(variable_name="historial"),
        (
            "human",
            """
Contexto recuperado de los documentos:

{contexto}

Pregunta actual:

{pregunta}
            """
        )
    ]
)

print("Prompt conversacional creado correctamente.")

Prompt conversacional creado correctamente.


In [39]:
def responder_pregunta_con_memoria(
    pregunta: str,
    session_id: str = "sesion-principal"
) -> dict:
    """
    Responde preguntas sobre los PDF conservando el historial
    de conversación de cada sesión.
    """

    if not pregunta or not pregunta.strip():
        return {
            "pregunta_original": pregunta,
            "pregunta_reformulada": "",
            "respuesta": "Debe ingresar una pregunta.",
            "fuentes": [],
            "documentos": []
        }

    pregunta = pregunta.strip()

    # 1. Recuperar el historial de la sesión
    historial_sesion = obtener_historial(session_id)
    mensajes_anteriores = historial_sesion.messages

    # 2. Reformular la pregunta usando el historial
    pregunta_reformulada = reformular_pregunta(
        pregunta=pregunta,
        historial=mensajes_anteriores
    )

    # 3. Buscar en FAISS con la pregunta completa
    documentos_recuperados = retriever.invoke(
        pregunta_reformulada
    )

    # 4. Preparar el contexto
    contexto = formatear_contexto(
        documentos_recuperados
    )

    # 5. Construir el prompt con historial
    mensajes_prompt = prompt_rag_con_memoria.invoke(
        {
            "historial": mensajes_anteriores,
            "contexto": contexto,
            "pregunta": pregunta
        }
    )

    # 6. Generar la respuesta
    resultado = llm.invoke(mensajes_prompt)
    respuesta = resultado.content.strip()

    # 7. Guardar pregunta y respuesta en el historial
    historial_sesion.add_user_message(pregunta)
    historial_sesion.add_ai_message(respuesta)

    # 8. Preparar fuentes sin duplicados
    fuentes = []

    for documento in documentos_recuperados:
        archivo = documento.metadata.get(
            "source",
            "Documento desconocido"
        )

        pagina = documento.metadata.get(
            "pagina_real",
            documento.metadata.get("page", 0) + 1
        )

        fuente = {
            "archivo": archivo,
            "pagina": pagina
        }

        if fuente not in fuentes:
            fuentes.append(fuente)

    return {
        "pregunta_original": pregunta,
        "pregunta_reformulada": pregunta_reformulada,
        "respuesta": respuesta,
        "fuentes": fuentes,
        "documentos": documentos_recuperados,
        "session_id": session_id
    }

In [40]:
def preguntar_al_agente_con_memoria(
    pregunta: str,
    session_id: str = "sesion-principal"
) -> dict:

    resultado = responder_pregunta_con_memoria(
        pregunta=pregunta,
        session_id=session_id
    )

    print("=" * 90)
    print("PREGUNTA ORIGINAL")
    print("=" * 90)
    print(resultado["pregunta_original"])

    print("\n" + "=" * 90)
    print("PREGUNTA UTILIZADA PARA BUSCAR EN FAISS")
    print("=" * 90)
    print(resultado["pregunta_reformulada"])

    print("\n" + "=" * 90)
    print("RESPUESTA")
    print("=" * 90)
    print(resultado["respuesta"])

    print("\n" + "=" * 90)
    print("FUENTES RECUPERADAS")
    print("=" * 90)

    if resultado["fuentes"]:
        for fuente in resultado["fuentes"]:
            print(
                f"- {fuente['archivo']}, "
                f"página {fuente['pagina']}"
            )
    else:
        print("No se recuperaron fuentes.")

    return resultado

In [41]:
almacen_conversaciones.clear()

print("Todas las conversaciones fueron reiniciadas.")

Todas las conversaciones fueron reiniciadas.


In [42]:
def mostrar_historial(
    session_id: str = "sesion-principal"
) -> None:

    historial = obtener_historial(session_id)

    print(f"HISTORIAL DE LA SESIÓN: {session_id}")
    print("=" * 90)

    if not historial.messages:
        print("La conversación está vacía.")
        return

    for mensaje in historial.messages:

        if isinstance(mensaje, HumanMessage):
            rol = "USUARIO"

        elif isinstance(mensaje, AIMessage):
            rol = "ASISTENTE"

        else:
            rol = mensaje.__class__.__name__

        print(f"\n{rol}:")
        print(mensaje.content)

In [43]:
mostrar_historial("prueba-retcc")

HISTORIAL DE LA SESIÓN: prueba-retcc
La conversación está vacía.


In [44]:
def reiniciar_conversacion(
    session_id: str = "sesion-principal"
) -> None:

    if session_id in almacen_conversaciones:
        almacen_conversaciones[session_id].clear()

    print(
        f"Conversación '{session_id}' "
        "reiniciada correctamente."
    )

In [45]:
import gradio as gr

print("Versión de Gradio:", gr.__version__)

Versión de Gradio: 5.49.1


In [46]:
reiniciar_conversacion("prueba-retcc")


Conversación 'prueba-retcc' reiniciada correctamente.


In [47]:
print("Versión de Gradio:", gr.__version__)

Versión de Gradio: 5.49.1


In [48]:
#------------------------------------------
#función para conectar Gradio con el agente
#------------------------------------------
import uuid

In [49]:
def crear_mensaje(rol: str, texto: str) -> dict:
    return {
        "role": rol,
        "content": texto
    }

In [50]:
import traceback
import uuid


def procesar_mensaje(
    mensaje: str,
    historial_chat: list,
    session_id: str
):
    if not mensaje or not mensaje.strip():
        return "", historial_chat or [], session_id

    if historial_chat is None:
        historial_chat = []

    if not session_id:
        session_id = str(uuid.uuid4())

    pregunta = mensaje.strip()

    historial_actualizado = list(historial_chat)

    historial_actualizado.append(
        crear_mensaje(
            "user",
            pregunta
        )
    )

    try:
        resultado = responder_pregunta_con_memoria(
            pregunta=pregunta,
            session_id=session_id
        )

        respuesta = resultado.get(
            "respuesta",
            "No se pudo generar una respuesta."
        )

        historial_actualizado.append(
            crear_mensaje(
                "assistant",
                respuesta
            )
        )

    except Exception:

        print("\n" + "=" * 90)
        print("ERROR COMPLETO DEL CHATBOT")
        print("=" * 90)

        traceback.print_exc()

        historial_actualizado.append(
            crear_mensaje(
                "assistant",
                (
                    "No fue posible procesar la consulta en este momento. "
                    "Por favor, inténtelo nuevamente."
                )
            )
        )

    return "", historial_actualizado, session_id

In [51]:
# ---------------------------------------------
# Función para iniciar una conversación nueva
# ---------------------------------------------

def nueva_conversacion(session_id: str):

    if session_id:
        try:
            reiniciar_conversacion(session_id)
        except Exception as error:
            print(
                "No se pudo reiniciar la conversación:",
                repr(error)
            )

    nuevo_session_id = str(uuid.uuid4())

    historial_inicial = [
        crear_mensaje(
            "assistant",
            (
                "¡Hola! Soy el asistente virtual del RETCC. "
                "Puedo ayudarte con consultas sobre inscripción, "
                "renovación, requisitos y procedimientos."
            )
        )
    ]

    return historial_inicial, nuevo_session_id, ""

In [52]:
def crear_mensaje(rol: str, texto: str) -> dict:
    return {
        "role": rol,
        "content": texto
    }

In [54]:
# ---------------------------------------------------------
# INTERFAZ GRÁFICA RETCC - RESPONSIVE
# Compatible con Gradio 5.49.1
# ---------------------------------------------------------

CSS_RETCC = """
:root {
    --retcc-red: #c60000;
    --retcc-red-dark: #9f0000;
    --retcc-red-soft: #fff1f1;
    --retcc-border: #ecd0d0;
    --retcc-text: #2b2b2b;
    --retcc-muted: #6f6f6f;
    --retcc-bg: #f5f6f8;
    --retcc-white: #ffffff;
}

html,
body {
    background: var(--retcc-bg) !important;
}

.gradio-container {
    max-width: 100% !important;
    min-height: 100vh;
    padding: 0 !important;
    background: var(--retcc-bg) !important;
    font-family: Arial, Helvetica, sans-serif;
    color: var(--retcc-text) !important;
}

#app-retcc {
    max-width: 1450px;
    margin: 0 auto;
    padding: 16px;
}

#cabecera-retcc {
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 16px;
    padding: 14px 20px;
    color: #ffffff !important;
    background: linear-gradient(
        90deg,
        var(--retcc-red-dark),
        var(--retcc-red)
    );
    border-radius: 16px 16px 0 0;
    box-shadow: 0 8px 20px rgba(120, 0, 0, 0.16);
}

#cabecera-retcc * {
    color: #ffffff !important;
}

.cabecera-marca {
    display: flex;
    align-items: center;
    gap: 12px;
}

.casquito-retcc {
    width: 48px;
    height: 48px;
    flex: 0 0 auto;
    padding: 5px;
    background: #ffffff;
    border-radius: 12px;
}

.casquito-retcc svg {
    width: 100%;
    height: 100%;
    display: block;
}

.cabecera-titulo {
    margin: 0;
    font-size: 22px;
    font-weight: 800;
}

.cabecera-subtitulo {
    margin: 3px 0 0;
    font-size: 12px;
    opacity: 0.95;
}

.cabecera-info {
    width: 28px;
    height: 28px;
    display: grid;
    place-items: center;
    border: 2px solid rgba(255, 255, 255, 0.9);
    border-radius: 50%;
    font-weight: bold;
}

#estado-retcc {
    padding: 9px 20px;
    color: #666666 !important;
    background: #ffffff !important;
    border-left: 1px solid var(--retcc-border);
    border-right: 1px solid var(--retcc-border);
    border-bottom: 1px solid var(--retcc-border);
    font-size: 12px;
}

#estado-retcc strong {
    color: var(--retcc-red) !important;
}

#zona-principal {
    gap: 0 !important;
    background: #ffffff !important;
    border: 1px solid var(--retcc-border);
    border-top: 0;
    border-radius: 0 0 16px 16px;
    overflow: hidden;
    box-shadow: 0 10px 26px rgba(0, 0, 0, 0.08);
}

/* =====================================================
   PANEL LATERAL
   ===================================================== */

#panel-lateral {
    min-height: 690px;
    padding: 18px !important;
    color: #2b2b2b !important;
    background: #ffffff !important;
    border-right: 1px solid var(--retcc-border);
}

#panel-lateral,
#panel-lateral div,
#panel-lateral p,
#panel-lateral span,
#panel-lateral strong {
    color: #2b2b2b !important;
    opacity: 1 !important;
}

.titulo-lateral {
    margin: 0 0 14px;
    color: var(--retcc-red) !important;
    font-size: 13px;
    font-weight: 800;
    letter-spacing: 0.4px;
}

.recurso-item {
    display: flex;
    align-items: center;
    gap: 10px;
    margin: 4px 0;
    padding: 11px 10px;
    color: #333333 !important;
    background: transparent;
    border-radius: 9px;
    font-size: 13px;
}

.recurso-item span {
    color: #333333 !important;
}

.recurso-item:hover {
    background: var(--retcc-red-soft) !important;
}

.recurso-icono {
    width: 28px;
    height: 28px;
    display: grid;
    place-items: center;
    color: var(--retcc-red) !important;
    background: #ffffff !important;
    border: 1px solid var(--retcc-border);
    border-radius: 8px;
    font-size: 14px;
}

.ayuda-box {
    margin-top: 26px;
    padding: 14px;
    color: #555555 !important;
    background: var(--retcc-red-soft) !important;
    border: 1px solid var(--retcc-border);
    border-radius: 12px;
    font-size: 12px;
}

.ayuda-box strong {
    color: var(--retcc-red-dark) !important;
}

.ayuda-box p {
    color: #555555 !important;
}

.usuario-box {
    display: flex;
    align-items: center;
    gap: 9px;
    margin-top: 18px;
    padding: 12px 4px 0;
    color: #444444 !important;
    border-top: 1px solid #eeeeee;
    font-size: 12px;
}

.usuario-box span,
.usuario-box strong {
    color: #444444 !important;
}

.usuario-avatar {
    width: 34px;
    height: 34px;
    display: grid;
    place-items: center;
    color: #333333 !important;
    background: #eeeeee !important;
    border-radius: 50%;
    font-size: 16px;
}

/* =====================================================
   ZONA DE CHAT
   ===================================================== */

#zona-chat {
    min-width: 0;
    padding: 18px !important;
    background: #fffafa !important;
}

#chat-retcc {
    min-height: 515px !important;
    color: #2b2b2b !important;
    background: #fffafa !important;
    border: 1px solid var(--retcc-border) !important;
    border-radius: 14px !important;
    box-shadow: none !important;
}

#chat-retcc > div,
#chat-retcc .wrap,
#chat-retcc .bubble-wrap,
#chat-retcc .message-wrap {
    background: #fffafa !important;
}

#chat-retcc .message {
    border-radius: 12px !important;
    font-size: 14px !important;
    line-height: 1.55 !important;
}

/* Mensajes del asistente */
#chat-retcc .message.bot,
#chat-retcc .bot,
#chat-retcc [data-testid="bot"] {
    color: #2b2b2b !important;
    background: #ffffff !important;
    border: 1px solid var(--retcc-border) !important;
}

#chat-retcc .message.bot *,
#chat-retcc .bot *,
#chat-retcc [data-testid="bot"] * {
    color: #2b2b2b !important;
    opacity: 1 !important;
}

/* Mensajes del usuario */
#chat-retcc .message.user,
#chat-retcc .user,
#chat-retcc [data-testid="user"] {
    color: #ffffff !important;
    background: var(--retcc-red) !important;
    border: 1px solid var(--retcc-red) !important;
}

#chat-retcc .message.user *,
#chat-retcc .user *,
#chat-retcc [data-testid="user"] * {
    color: #ffffff !important;
    opacity: 1 !important;
}

/* =====================================================
   CONSULTAS FRECUENTES
   ===================================================== */

.preguntas-rapidas {
    margin: 10px 0 8px;
    color: #555555 !important;
    font-size: 12px;
    font-weight: 700;
}

.boton-rapido {
    color: var(--retcc-red-dark) !important;
    background: #ffffff !important;
    border: 1px solid #db5b5b !important;
    border-radius: 999px !important;
    font-size: 12px !important;
}

.boton-rapido *,
.boton-rapido span {
    color: var(--retcc-red-dark) !important;
    opacity: 1 !important;
}

.boton-rapido:hover {
    color: #ffffff !important;
    background: var(--retcc-red) !important;
}

.boton-rapido:hover *,
.boton-rapido:hover span {
    color: #ffffff !important;
}

/* =====================================================
   CAJA DE TEXTO Y BOTONES
   ===================================================== */

#entrada-mensaje {
    background: #ffffff !important;
    border-radius: 12px !important;
}

#entrada-mensaje textarea {
    min-height: 55px !important;
    color: #2b2b2b !important;
    background: #ffffff !important;
    caret-color: var(--retcc-red) !important;
    border-radius: 14px !important;
    font-size: 14px !important;
}

#entrada-mensaje textarea::placeholder {
    color: #888888 !important;
    opacity: 1 !important;
}

#boton-enviar {
    min-width: 120px;
    height: 56px;
    color: #ffffff !important;
    background: var(--retcc-red) !important;
    border: 0 !important;
    border-radius: 12px !important;
    font-weight: 700 !important;
}

#boton-enviar * {
    color: #ffffff !important;
}

#boton-enviar:hover {
    background: var(--retcc-red-dark) !important;
}

#boton-nuevo {
    width: 100%;
    margin-top: 16px;
    color: var(--retcc-red) !important;
    background: #ffffff !important;
    border: 1px solid var(--retcc-red) !important;
    border-radius: 10px !important;
}

#boton-nuevo * {
    color: var(--retcc-red) !important;
}

#boton-soporte {
    width: 100%;
    margin-top: 10px;
    color: #ffffff !important;
    background: var(--retcc-red) !important;
    border: 0 !important;
    border-radius: 8px !important;
}

#boton-soporte * {
    color: #ffffff !important;
}

#pie-retcc {
    padding: 10px 4px 0;
    color: #777777 !important;
    text-align: center;
    font-size: 10px;
    letter-spacing: 0.3px;
}

/* =====================================================
   RESPONSIVE
   ===================================================== */

@media (max-width: 900px) {
    #app-retcc {
        padding: 0;
    }

    #cabecera-retcc {
        border-radius: 0;
    }

    #zona-principal {
        border-radius: 0;
    }

    #panel-lateral {
        min-height: auto;
        border-right: 0;
        border-bottom: 1px solid var(--retcc-border);
    }

    #chat-retcc {
        min-height: 430px !important;
    }

    .cabecera-subtitulo {
        display: none;
    }
}

@media (max-width: 600px) {
    #cabecera-retcc {
        padding: 12px;
    }

    .casquito-retcc {
        width: 40px;
        height: 40px;
    }

    .cabecera-titulo {
        font-size: 17px;
    }

    #zona-chat,
    #panel-lateral {
        padding: 12px !important;
    }

    #boton-enviar {
        min-width: 82px;
    }

    #chat-retcc {
        min-height: 390px !important;
    }
}
"""

CASCO_SVG = """
<svg
    viewBox="0 0 128 128"
    xmlns="http://www.w3.org/2000/svg"
    aria-label="Casco de construcción"
>
    <path
        d="M23 72c1-29 18-48 41-48s40 19 41 48"
        fill="#ffd52a"
        stroke="#1e1e1e"
        stroke-width="5"
    />
    <path
        d="M52 27v43M76 27v43"
        stroke="#1e1e1e"
        stroke-width="5"
    />
    <path
        d="M16 73h96c5 0 9 4 9 9v5H7v-5c0-5 4-9 9-9Z"
        fill="#ffd52a"
        stroke="#1e1e1e"
        stroke-width="5"
    />
    <path
        d="M27 88h74"
        stroke="#1e1e1e"
        stroke-width="5"
        stroke-linecap="round"
    />
</svg>
"""

MENSAJE_INICIAL = [
    {
        "role": "assistant",
        "content": (
            "¡Hola! Soy tu asistente virtual del Registro Nacional de "
            "Trabajadores de Construcción Civil (RETCC).\n\n"
            "Puedo ayudarte con información sobre inscripción, renovación, "
            "requisitos, plazos y procedimientos. "
            "¿En qué puedo orientarte?"
        )
    }
]


with gr.Blocks(
    title="Asistente de Consultas RETCC",
    css=CSS_RETCC,
    theme=gr.themes.Soft(
        primary_hue="red",
        neutral_hue="gray"
    )
) as interfaz_retcc:

    session_id = gr.State(value="")

    with gr.Column(elem_id="app-retcc"):

        gr.HTML(
            f"""
            <header id="cabecera-retcc">
                <div class="cabecera-marca">
                    <div class="casquito-retcc">
                        {CASCO_SVG}
                    </div>

                    <div>
                        <h1 class="cabecera-titulo">
                            RETCC Chatbot
                        </h1>

                        <p class="cabecera-subtitulo">
                            Asistente oficial de consultas del Registro
                            Nacional de Trabajadores de Construcción Civil
                        </p>
                    </div>
                </div>

                <div class="cabecera-info">i</div>
            </header>

            <div id="estado-retcc">
                Estado de consulta:
                <strong>Disponible</strong>
            </div>
            """
        )

        with gr.Row(elem_id="zona-principal"):

            with gr.Column(
                scale=1,
                min_width=245,
                elem_id="panel-lateral"
            ):

                gr.HTML(
                    """
                    <p class="titulo-lateral">
                        RECURSOS OFICIALES
                    </p>

                    <div class="recurso-item">
                        <span class="recurso-icono">▤</span>
                        <span>Guía de inscripción</span>
                    </div>

                    <div class="recurso-item">
                        <span class="recurso-icono">✓</span>
                        <span>Reglamento RETCC</span>
                    </div>

                    <div class="recurso-item">
                        <span class="recurso-icono">◷</span>
                        <span>Requisitos y plazos</span>
                    </div>

                    <div class="ayuda-box">
                        <strong>¿Necesitas ayuda?</strong>

                        <p>
                            Nuestro equipo de soporte puede orientarte
                            cuando la información no esté disponible.
                        </p>
                    </div>

                    <div class="usuario-box">
                        <span class="usuario-avatar">👤</span>

                        <span>
                            <strong>Ciudadano</strong><br>
                            Usuario del servicio RETCC
                        </span>
                    </div>
                    """
                )

                boton_soporte = gr.Button(
                    "Contactar soporte",
                    elem_id="boton-soporte"
                )

                boton_nuevo = gr.Button(
                    "Nueva conversación",
                    elem_id="boton-nuevo"
                )

            with gr.Column(
                scale=4,
                min_width=0,
                elem_id="zona-chat"
            ):

                chatbot = gr.Chatbot(
                    value=MENSAJE_INICIAL,
                    type="messages",
                    elem_id="chat-retcc",
                    show_label=False,
                    height=515,
                    avatar_images=(None, None)
                )

                gr.HTML(
                    """
                    <div class="preguntas-rapidas">
                        CONSULTAS FRECUENTES
                    </div>
                    """
                )

                with gr.Row():

                    btn_requisitos = gr.Button(
                        "Requisitos para inscripción",
                        elem_classes=["boton-rapido"]
                    )

                    btn_renovacion = gr.Button(
                        "¿Cómo renovar mi carné?",
                        elem_classes=["boton-rapido"]
                    )

                    btn_plazo = gr.Button(
                        "¿Cuánto demora el trámite?",
                        elem_classes=["boton-rapido"]
                    )

                with gr.Row():

                    entrada_mensaje = gr.Textbox(
                        placeholder=(
                            "Escribe tu consulta sobre el RETCC..."
                        ),
                        show_label=False,
                        lines=1,
                        max_lines=3,
                        elem_id="entrada-mensaje",
                        scale=8
                    )

                    boton_enviar = gr.Button(
                        "Enviar",
                        elem_id="boton-enviar",
                        scale=1
                    )

        gr.HTML(
            """
            <footer id="pie-retcc">
                SISTEMA DE ORIENTACIÓN DEL REGISTRO NACIONAL DE
                TRABAJADORES DE CONSTRUCCIÓN CIVIL
            </footer>
            """
        )

    evento_boton = boton_enviar.click(
        fn=procesar_mensaje,
        inputs=[
            entrada_mensaje,
            chatbot,
            session_id
        ],
        outputs=[
            entrada_mensaje,
            chatbot,
            session_id
        ],
      show_progress="minimal"
    )

    evento_enter = entrada_mensaje.submit(
        fn=procesar_mensaje,
        inputs=[
            entrada_mensaje,
            chatbot,
            session_id
        ],
        outputs=[
            entrada_mensaje,
            chatbot,
            session_id
        ]
    )

    boton_nuevo.click(
        fn=nueva_conversacion,
        inputs=[session_id],
        outputs=[
            chatbot,
            session_id,
            entrada_mensaje
        ],
    show_progress="minimal"
    )

    btn_requisitos.click(
        fn=lambda: (
            "¿Cuáles son los requisitos para "
            "inscribirme en el RETCC?"
        ),
        outputs=entrada_mensaje
    )

    btn_renovacion.click(
        fn=lambda: (
            "¿Cómo puedo renovar mi carné RETCC?"
        ),
        outputs=entrada_mensaje
    )

    btn_plazo.click(
        fn=lambda: (
            "¿Cuánto demora el trámite del RETCC?"
        ),
        outputs=entrada_mensaje
    )

    boton_soporte.click(
        fn=lambda: (
            "Necesito orientación adicional porque "
            "no encontré respuesta a mi consulta."
        ),
        outputs=entrada_mensaje
    )

In [ ]:
interfaz_retcc.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fadb5ebdf28fc578d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Conversación '808f3642-0304-462e-a038-d732f9264505' reiniciada correctamente.


In [61]:
try:
    interfaz_retcc.close()
except Exception:
    pass

Closing server running on port: 7860
